# 04 Game Simulator

This notebook creates a possession-based game feed, then maps possessions onto game-clock timestamps.

The simulator combines team-level tendencies with player-level event weights: team profiles determine the pace and shot environment, while player weights decide who shoots, assists, rebounds, turns it over, and fills the box score.


## Load Team Profiles And Player Weights

The simulator needs both sides of the project: team-game rows for pace and efficiency, and player event weights for realistic individual box-score events.


In [ ]:
from pathlib import Path
import os
import unicodedata
import numpy as np
import pandas as pd

# Resolve paths so the notebook works from either the repo root or notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
WAREHOUSE = ROOT / 'warehouse'
MODELS = ROOT / 'models'
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.matplotlib'))
(ROOT / '.matplotlib').mkdir(exist_ok=True)

# Team data controls game shape: scoring level, shot mix, turnovers, and pace.
team = pd.read_csv(WAREHOUSE / 'fact_team_game.csv')
team['GAME_DATE'] = pd.to_datetime(team['GAME_DATE'])

weights_path = MODELS / 'player_event_weights.csv'
# Player weights control event assignment inside that team-level game shape.
if weights_path.exists():
    player_weights = pd.read_csv(weights_path)
else:
    raise FileNotFoundError('Run 03_player_event_weights.ipynb first.')

team.tail()


## Helper Functions

These helpers summarize team tendencies, normalize names, sample players from weighted event pools, and format the clock. This mirrors the later `app/simulator.py` module.


In [ ]:
# Cache weighted player pools so repeated simulations do not rebuild them.
PLAYER_POOLS = {}

def team_profile(team_abbr, season=None, last_n=25):
    # Summarize recent team tendencies that drive possession outcomes.
    data = team[team['TEAM_ABBREVIATION'] == team_abbr].sort_values('GAME_DATE')
    if season is not None:
        data = data[data['SEASON'] == season]
    data = data.tail(last_n)
    return {
        'pts': data['PTS'].mean(),
        'fga': data['FGA'].mean(),
        'fg_pct': data['FG_PCT'].mean(),
        'fg3_rate': (data['FG3A'].sum() / max(data['FGA'].sum(), 1)),
        'fg3_pct': data['FG3_PCT'].mean(),
        'fta_rate': (data['FTA'].sum() / max(data['FGA'].sum(), 1)),
        'ft_pct': data['FT_PCT'].mean(),
        'tov_rate': (data['TOV'].sum() / max(data['FGA'].sum() + data['FTA'].sum() + data['TOV'].sum(), 1)),
        'oreb': data['OREB'].mean(),
        'dreb': data['DREB'].mean(),
        'ast': data['AST'].mean(),
    }

def ascii_name(value):
    # Normalize accented names for clean notebook output and browser display.
    return unicodedata.normalize('NFKD', str(value)).encode('ascii', 'ignore').decode('ascii')

def choose_player(team_abbr, weight_col, rng):
    # Sample a player from a team-specific event distribution, such as scoring_weight.
    key = (team_abbr, weight_col)
    if key not in PLAYER_POOLS:
        pool = player_weights[player_weights['TEAM_ABBR'] == team_abbr].copy()
        pool = pool[pool[weight_col] > 0]
        if pool.empty:
            PLAYER_POOLS[key] = None
        else:
            availability = pool.get('availability_weight', 1.0)
            probs = pool[weight_col].to_numpy(dtype=float) * np.asarray(availability, dtype=float)
            probs = probs / probs.sum()
            meta_cols = ['expected_min', 'player_fg_pct', 'player_fg3_pct', 'player_ft_pct']
            for col in meta_cols:
                if col not in pool.columns:
                    pool[col] = np.nan
            meta = pool[meta_cols].to_dict('records')
            PLAYER_POOLS[key] = (pool['PLAYER_NAME'].map(ascii_name).to_numpy(), probs, meta)
    if PLAYER_POOLS[key] is None:
        return team_abbr
    names, probs, _ = PLAYER_POOLS[key]
    return rng.choice(names, p=probs)

def choose_player_profile(team_abbr, weight_col, rng):
    name = choose_player(team_abbr, weight_col, rng)
    pool = player_weights[player_weights['TEAM_ABBR'] == team_abbr].copy()
    pool['PLAYER_NAME_ASCII'] = pool['PLAYER_NAME'].map(ascii_name)
    match = pool[pool['PLAYER_NAME_ASCII'] == name]
    profile = match.iloc[0].to_dict() if not match.empty else {}
    return name, profile

def expected_minutes_lookup(team_abbr):
    # Expected minutes are used only for box-score display, not winner prediction.
    pool = player_weights[player_weights['TEAM_ABBR'] == team_abbr].copy()
    if 'expected_min' not in pool.columns:
        return {}
    pool['PLAYER_NAME'] = pool['PLAYER_NAME'].map(ascii_name)
    return dict(zip(pool['PLAYER_NAME'], pool['expected_min']))

def format_clock(seconds_remaining):
    quarter = 5 - int(np.ceil(seconds_remaining / 720))
    quarter = min(max(quarter, 1), 4)
    q_elapsed = 720 - ((seconds_remaining - 1) % 720 + 1)
    q_remaining = 720 - q_elapsed
    minutes = int(q_remaining // 60)
    seconds = int(q_remaining % 60)
    return f'Q{quarter} {minutes:02d}:{seconds:02d}'


## Simulate One Game

The game loop alternates possessions, samples turnovers/free throws/field goals, assigns events to weighted players, updates the score, and builds a play-by-play feed plus player box score.


In [ ]:
def simulate_game(home_team, away_team, season=None, seed=7, verbose=True):
    rng = np.random.default_rng(seed)
    home_profile = team_profile(home_team, season)
    away_profile = team_profile(away_team, season)

    # Estimate possessions from recent scoring, then clip to a normal NBA range.
    expected_total = np.nanmean([home_profile['pts'], away_profile['pts']]) * 2
    possessions = int(np.clip(round(expected_total / 1.12), 176, 212))
    seconds_remaining = 48 * 60
    score = {home_team: 0, away_team: 0}
    feed = []
    stat_cols = ['PTS', 'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA', 'OREB', 'REB', 'AST', 'TOV']
    box = {}
    expected_mins = {**expected_minutes_lookup(home_team), **expected_minutes_lookup(away_team)}

    def ensure_player(team_abbr, player_name):
        # Create each player row lazily when that player first appears.
        key = (team_abbr, player_name)
        if key not in box:
            box[key] = {'TEAM': team_abbr, 'PLAYER': player_name, 'MIN': expected_mins.get(player_name, 0.0), **{col: 0 for col in stat_cols}}
        return box[key]

    def add_stat(team_abbr, player_name, **stats):
        row = ensure_player(team_abbr, player_name)
        for stat, value in stats.items():
            row[stat] += value

    for possession in range(possessions):
        offense = home_team if possession % 2 == 0 else away_team
        profile = home_profile if offense == home_team else away_profile
        clock = format_clock(seconds_remaining)

        # Turnovers end possessions before shot attempts.
        if rng.random() < profile['tov_rate']:
            player = choose_player(offense, 'turnover_weight', rng)
            add_stat(offense, player, TOV=1)
            feed.append((clock, offense, f'{player} turnover'))
        else:
            # Free throws use team rate for frequency and player FT% for makes.
            if rng.random() < min(profile['fta_rate'] * 0.35, 0.16):
                shooter, shooter_profile = choose_player_profile(offense, 'free_throw_weight', rng)
                ft_pct = shooter_profile.get('player_ft_pct', profile.get('ft_pct', 0.77))
                ft_pct = 0.77 if pd.isna(ft_pct) else float(ft_pct)
                made_fts = int(rng.binomial(2, ft_pct))
                score[offense] += made_fts
                add_stat(offense, shooter, PTS=made_fts, FTM=made_fts, FTA=2)
                feed.append((clock, offense, f'{shooter} makes {made_fts} of 2 free throws'))
                seconds_remaining = max(0, seconds_remaining - int(rng.integers(12, 26)))
                continue

            # Team three-point rate decides shot type; player weights decide shooter.
            is_three = rng.random() < profile['fg3_rate']
            shot_weight = 'three_weight' if is_three else 'scoring_weight'
            shooter, shooter_profile = choose_player_profile(offense, shot_weight, rng)
            if is_three:
                player_make = shooter_profile.get('player_fg3_pct', profile['fg3_pct'])
                make_prob = 0.55 * profile['fg3_pct'] + 0.45 * player_make
            else:
                player_make = shooter_profile.get('player_fg_pct', profile['fg_pct'])
                make_prob = min(0.55 * (profile['fg_pct'] + 0.06) + 0.45 * player_make, 0.68)
            make_prob = 0.45 if pd.isna(make_prob) else float(make_prob)
            made = rng.random() < make_prob

            if made:
                points = 3 if is_three else 2
                score[offense] += points
                assist = choose_player(offense, 'assist_weight', rng)
                shot_name = '3PT shot' if is_three else '2PT shot'
                add_stat(offense, shooter, PTS=points, FGM=1, FGA=1, FG3M=int(is_three), FG3A=int(is_three))
                if assist != shooter and rng.random() < 0.62:
                    add_stat(offense, assist, AST=1)
                    feed.append((clock, offense, f'{shooter} makes {shot_name} assisted by {assist}'))
                else:
                    feed.append((clock, offense, f'{shooter} makes {shot_name}'))
            else:
                shot_name = '3PT shot' if is_three else '2PT shot'
                add_stat(offense, shooter, FGA=1, FG3A=int(is_three))
                feed.append((clock, offense, f'{shooter} misses {shot_name}'))

                # Misses become rebound events assigned through rebound weights.
                if rng.random() < 0.24:
                    rebounder = choose_player(offense, 'rebound_weight', rng)
                    add_stat(offense, rebounder, OREB=1, REB=1)
                    feed.append((clock, offense, f'{rebounder} offensive rebound'))
                else:
                    defense = away_team if offense == home_team else home_team
                    rebounder = choose_player(defense, 'rebound_weight', rng)
                    add_stat(defense, rebounder, REB=1)

        seconds_remaining = max(0, seconds_remaining - int(rng.integers(7, 25)))
        if seconds_remaining <= 0:
            break

    if verbose:
        print(f'{away_team} {score[away_team]} @ {home_team} {score[home_team]}')
        print()
        for clock, team_abbr, text in feed[:80]:
            print(f'{clock} - {team_abbr}: {text}')

    box_score = pd.DataFrame(box.values())
    if not box_score.empty:
        box_score['FG'] = box_score['FGM'].astype(str) + '-' + box_score['FGA'].astype(str)
        box_score['FG3'] = box_score['FG3M'].astype(str) + '-' + box_score['FG3A'].astype(str)
        box_score['FT'] = box_score['FTM'].astype(str) + '-' + box_score['FTA'].astype(str)
        box_score = box_score.sort_values(['TEAM', 'MIN', 'PTS'], ascending=[True, False, False])
        box_score = box_score[['TEAM', 'PLAYER', 'MIN', 'PTS', 'REB', 'AST', 'FG', 'FG3', 'FT', 'OREB', 'TOV']].reset_index(drop=True)
    return score, feed, box_score

score, feed, box_score = simulate_game('PHX', 'DEN', season='2025-26', seed=42)


## Single Game Box Score

Run this cell after the simulation cell above. It displays the final score and one box score table per team.

In [ ]:
home_team = 'PHX'
away_team = 'DEN'

print(f"Final: {away_team} {score[away_team]} @ {home_team} {score[home_team]}")

print(f"\n{away_team} Box Score")
display(box_score[box_score['TEAM'] == away_team].reset_index(drop=True))

print(f"\n{home_team} Box Score")
display(box_score[box_score['TEAM'] == home_team].reset_index(drop=True))

## Run Many Simulations

Repeating the same matchup across many seeds gives a distribution of possible scores and margins. This is useful for exploring the simulator's behavior, but it is separate from the trained Random Forest model's held-out accuracy.


In [ ]:
def simulate_many(home_team, away_team, season=None, n=1000):
        # Run many seeded simulations to estimate a matchup distribution.
    rows = []
    for seed in range(n):
        score, _, _ = simulate_game(home_team, away_team, season=season, seed=seed, verbose=False)
        rows.append({'home_team': home_team, 'away_team': away_team, 'home_score': score[home_team], 'away_score': score[away_team]})
    sims = pd.DataFrame(rows)
        # This is simulation-based home-win rate, not the Random Forest test accuracy.
    sims['home_win'] = sims['home_score'] > sims['away_score']
    sims['margin'] = sims['home_score'] - sims['away_score']
    return sims

sims = simulate_many('PHX', 'DEN', season='2025-26', n=1000)
sims[['home_score', 'away_score', 'margin']].describe()


## Summarize Simulation Distribution

These summary values describe the simulated matchup distribution: home win rate, average score, and spread of margins.


In [ ]:
print('Home win probability:', round(sims['home_win'].mean(), 3))
print('Average score:', round(sims['home_score'].mean(), 1), '-', round(sims['away_score'].mean(), 1))
sims['margin'].hist(bins=25)